# Parsers
> Convert raw file bytes into plain text for chunking and ingestion.

Anchor parsers implement a simple interface: `parse(content: bytes) -> str`.
Each parser declares which file extensions it supports via
`supported_extensions`.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.ingestion.parsers import PlainTextParser, MarkdownParser, HTMLParser

## PlainTextParser
The simplest parser -- decodes bytes to a UTF-8 string with optional
whitespace normalization.

In [ ]:
text_parser = PlainTextParser()

print(f"Supported extensions: {text_parser.supported_extensions}")

raw = b"Hello, world!\nThis is a plain text document.\n"
result = text_parser.parse(raw)

print(f"\nInput bytes: {len(raw)} bytes")
print(f"Output text: {repr(result)}")

## MarkdownParser
Strips Markdown formatting (headings, bold, links, etc.) and returns
clean text suitable for chunking.

In [ ]:
md_parser = MarkdownParser()

print(f"Supported extensions: {md_parser.supported_extensions}")

markdown_content = b"""# Project README

## Overview

This is a **sample** project that demonstrates the `Anchor` framework.

### Features

- Context-aware AI pipelines
- Modular [architecture](https://example.com)
- Built-in memory management

## Installation

```bash
pip install anchor
```

See the [docs](https://docs.example.com) for more details.
"""

result = md_parser.parse(markdown_content)

print(f"\nInput bytes:  {len(markdown_content)} bytes")
print(f"Output chars: {len(result)} chars")
print(f"\nParsed output:")
print(result)

## HTMLParser
Extracts visible text from HTML documents, stripping tags, scripts,
and style blocks.

In [ ]:
html_parser = HTMLParser()

print(f"Supported extensions: {html_parser.supported_extensions}")

html_content = b"""<!DOCTYPE html>
<html>
<head><title>Sample Page</title></head>
<body>
  <h1>Welcome to Anchor</h1>
  <p>Anchor is a <strong>context-aware</strong> AI framework.</p>
  <ul>
    <li>Memory management</li>
    <li>Retrieval-augmented generation</li>
    <li>Intelligent context assembly</li>
  </ul>
  <script>console.log('ignored');</script>
  <footer><p>Copyright 2024</p></footer>
</body>
</html>
"""

result = html_parser.parse(html_content)

print(f"\nInput bytes:  {len(html_content)} bytes")
print(f"Output chars: {len(result)} chars")
print(f"\nParsed output:")
print(result)

## Parser Selection Pattern
Match a file extension to the right parser at runtime.

In [ ]:
# Build a registry of parsers
parsers = [PlainTextParser(), MarkdownParser(), HTMLParser()]

registry = {}
for parser in parsers:
    for ext in parser.supported_extensions:
        registry[ext] = parser

print("Parser registry:")
for ext, parser in sorted(registry.items()):
    print(f"  {ext:<6} -> {type(parser).__name__}")

In [ ]:
# Lookup by extension
def parse_file(filename: str, content: bytes) -> str:
    ext = '.' + filename.rsplit('.', 1)[-1] if '.' in filename else ''
    parser = registry.get(ext)
    if parser is None:
        raise ValueError(f"No parser for extension: {ext}")
    return parser.parse(content)

# Test with different file types
test_files = [
    ("notes.txt", b"Plain text notes."),
    ("readme.md", b"# Heading\nSome **bold** text."),
    ("index.html", b"<p>Hello <em>world</em></p>"),
]

for filename, content in test_files:
    parsed = parse_file(filename, content)
    print(f"  {filename:<14} -> {repr(parsed)}")

## Key Takeaways
- All parsers implement `parse(content: bytes) -> str`
- `supported_extensions` declares which file types a parser handles
- **PlainTextParser**: UTF-8 decode with whitespace cleanup
- **MarkdownParser**: strips formatting, returns clean text
- **HTMLParser**: extracts visible text, ignores scripts/styles
- Build a registry dict to dispatch by file extension at runtime

**Next:** [Metadata Enrichment](04_metadata_enrichment.ipynb)